In [1]:
import re

In [2]:
text = "Paul Newman was an American actor, but Paul Hollywood is a British TV Host. The name Paul is quite common."
# define um texto

In [3]:
pattern = r"Paul [A-Z]\w+"
# regra para que contenha um nome com letra maiuscula depois do Paul

In [4]:
matches = re.finditer(pattern, text)
# retorna os textos que estao dentro do padrao definido pela regra
for match in matches:
    print (match)

<re.Match object; span=(0, 11), match='Paul Newman'>
<re.Match object; span=(39, 53), match='Paul Hollywood'>


In [5]:
import spacy
from spacy.tokens import Span

In [6]:
nlp = spacy.blank("en")
# cria um pipe vazio
doc = nlp(text)
print (doc.ents)
original_ents = list(doc.ents)
# entidades antes da modificação
mwt_ents = []
for match in re.finditer(pattern, doc.text):
    # para cada entidade que corresponde a regra aplicada
    start, end = match.span()
    # pega a posição em caracteres de inicio e fim dentro do doc
    span = doc.char_span(start, end)
    # tenta converter em um span spaCy
    #
    if span is not None:
        # se da certo, adiciona a lista de entidades
        mwt_ents.append((span.start, span.end, span.text))
for ent in mwt_ents:
    # para cada entidade pega sua posição inicial, final e nome
    start, end, name = ent
    per_ent = Span(doc, start, end, label="PERSON")
    # rotula o span como pessoa
    original_ents.append(per_ent)
    # adiciona a variavel modificada a lista
doc.ents = original_ents
# coloca as entidades modificadas as do doc
for ent in doc.ents:
    print (ent.text, ent.label_)

()
Paul Newman PERSON
Paul Hollywood PERSON


In [7]:
print (mwt_ents)

[(0, 2, 'Paul Newman'), (8, 10, 'Paul Hollywood')]


In [8]:
from spacy.language import Language

@Language.component("paul_ner")
# delara um componente a partir de uma função
def paul_ner(doc):
    pattern = r"Paul [A-Z]\w+"
    # mesma regra de antes, nome que começa com letra maiuscula
    original_ents = list(doc.ents)
    mwt_ents = []
    for match in re.finditer(pattern, doc.text):
        start, end = match.span()
        # pega os caracteres de inicio e fim
        span = doc.char_span(start, end)
        if span is not None:
            mwt_ents.append((span.start, span.end, span.text))
    for ent in mwt_ents:
        start, end, name = ent
        per_ent = Span(doc, start, end, label="PERSON")
        original_ents.append(per_ent)
    doc.ents = original_ents
    return (doc)

In [9]:
nlp2 = spacy.blank("en")
# cria um novo pipeline
nlp2.add_pipe("paul_ner")
# adiciona a funcao ao pipeline

<function __main__.paul_ner(doc)>

In [10]:
doc2 = nlp2(text)
# cria um outro documento
print (doc2.ents)

(Paul Newman, Paul Hollywood)


In [11]:
from spacy.language import Language
from spacy.util import filter_spans
@Language.component("cinema_ner")
def cinema_ner(doc):
    pattern = r"Hollywood"
    original_ents = list(doc.ents)
    mwt_ents = []
    for match in re.finditer(pattern, doc.text):
        start, end = match.span()
        span = doc.char_span(start, end)
        if span is not None:
            mwt_ents.append((span.start, span.end, span.text))
    for ent in mwt_ents:
        start, end, name = ent
        per_ent = Span(doc, start, end, label="CINEMA")
        original_ents.append(per_ent)
    filtered = filter_spans(original_ents)
    doc.ents = filtered
    return (doc)
# mesmo jeito declarado como antes, porém agora aplicando a uma regra onde o Hollywood apareça

In [12]:
nlp3 = spacy.load("en_core_web_sm")
nlp3.add_pipe("cinema_ner")

<function __main__.cinema_ner(doc)>

In [13]:
doc3 = nlp3(text)
for ent in doc3.ents:
    print (ent.text, ent.label_)

Paul Newman PERSON
American NORP
Paul Hollywood PERSON
British NORP
Paul PERSON
